# 크라베오 AI 재학습 — v6 (정체성 데이터 비율 조정)
# Cravveo AI Retrain — v6 (rebalanced identity data ratio)

**배경 | Background:**
Day 041(v5)에서 정체성 데이터를 20개로 정리했으나, Obsidian 데이터 203개에 비해 너무 적어서(9%)
"크라베오 컴퍼니가 뭐야?" 같은 질문에 학습 신호이 묻혀 완전히 엉뚱한 내용(가짜 URL, 가짜 서비스)을 생성함 — 과소적합(underfitting).
In Day 041 (v5), the 20 identity samples were drowned out by 203 Obsidian samples (only 9%),
causing severe underfitting on identity questions — the model hallucinated fake URLs/services.

**변경 사항 | Changes:**
- 정체성 데이터 20개를 10배 복제 → 200개
- Obsidian 데이터 203개와 거의 1:1 비율로 맞춤
- Repeat the 20 identity samples 10x → 200
- Now roughly balanced 1:1 with the 203 Obsidian samples

**순서 | Steps:**
1. 설치
2. 모델 로드
3. LoRA 어댑터
4. 데이터셋 로드 + 복제 + 합치기
5. 학습
6. 테스트
7. HuggingFace 업로드
8. GGUF 변환 (q4_K_M)


In [ ]:
# 셀 1: 필수 라이브러리 설치
# Cell 1: Install required libraries
!pip install unsloth trl datasets


In [ ]:
# 셀 2: CUDA 오류 방지 + 모델 로드
# Cell 2: CUDA fix + model load
import ctypes, glob

paths = (
    glob.glob("/usr/local/lib/python*/dist-packages/nvidia/*/lib/libnvJitLink*.so*") +
    glob.glob("/usr/lib/**/libnvJitLink*.so*", recursive=True) +
    glob.glob("/usr/local/cuda*/**/libnvJitLink*.so*", recursive=True)
)
if paths:
    ctypes.CDLL(paths[0], mode=ctypes.RTLD_GLOBAL)
    print(f"CUDA fix 적용 | CUDA fix applied: {paths[0]}")

from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Llama-3.2-3B-Instruct",
    max_seq_length=512,
    load_in_4bit=True,
)
print("모델 로드 완료 | Model loaded")


In [ ]:
# 셀 3: LoRA 어댑터 설정
# Cell 3: LoRA adapter setup
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
)
print("LoRA 어댑터 적용 완료 | LoRA adapter applied")


In [ ]:
# 셀 4: 데이터셋 로드 + 복제 + 합치기
# Cell 4: Load datasets + oversample + combine
from datasets import load_dataset, Dataset, concatenate_datasets
import json

# --- 정체성 데이터셋 (HuggingFace, 20개) ---
existing_raw = load_dataset("cravveo/cravveo-briefing-dataset", split="train")

def convert_existing(example):
    return {"text": f"질문: {example['instruction']}\n답변: {example['output']}"}

existing = existing_raw.map(convert_existing, remove_columns=existing_raw.column_names)
print(f"정체성 데이터셋(원본): {len(existing)}개")

# --- 10배 복제해서 Obsidian(203개)과 비율 맞추기 ---
REPEAT = 10
existing_oversampled = concatenate_datasets([existing] * REPEAT)
print(f"정체성 데이터셋(복제 후): {len(existing_oversampled)}개")

# --- Obsidian 데이터셋 (파일 업로드) ---
from google.colab import files
print("obsidian_dataset.jsonl 파일을 업로드하세요 | Upload obsidian_dataset.jsonl")
uploaded = files.upload()

new_data = []
with open("obsidian_dataset.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        new_data.append(json.loads(line.strip()))

new_dataset = Dataset.from_list(new_data)
print(f"Obsidian 데이터셋: {len(new_dataset)}개")

# --- 합치기 ---
combined = concatenate_datasets([existing_oversampled, new_dataset])
combined = combined.shuffle(seed=42)
print(f"\n합계 | Total: {len(combined)}개 (정체성 {len(existing_oversampled)} : Obsidian {len(new_dataset)})")
print("샘플 확인 | Sample check:")
print(combined[0]['text'])


In [ ]:
# 셀 5: 학습
# Cell 5: Training
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=combined,
    dataset_text_field="text",
    max_seq_length=512,
    dataset_num_proc=2,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=True,
        logging_steps=10,
        output_dir="cravveo_v6_output",
        save_strategy="no",
    ),
)

print("학습 시작 | Training start...")
trainer.train()
print("학습 완료 | Training complete!")


In [ ]:
# 셀 6: 테스트 — 학습 결과 확인
# Cell 6: Quick test
FastLanguageModel.for_inference(model)

test_questions = [
    "크라베오 컴퍼니가 뭐야?",
    "ablescan.app은 뭐야?",
    "지금 수익이 나고 있어?",
    "Day 032의 목표가 뭐야?",
]
for q in test_questions:
    inputs = tokenizer([f"질문: {q}\n답변: "], return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=100, temperature=0.2)
    print(tokenizer.decode(outputs[0], skip_special_tokens=True))
    print("---")


In [ ]:
# 셀 7: HuggingFace 업로드
# Cell 7: Push to HuggingFace
from google.colab import userdata
from huggingface_hub import login

token = userdata.get('HF_TOKEN')
login(token=token)

model.save_pretrained("cravveo-llama-lora-v6")
model.push_to_hub("cravveo/cravveo-llama-lora", token=token)  # 같은 레포에 덮어쓰기
tokenizer.push_to_hub("cravveo/cravveo-llama-lora", token=token)

print("HuggingFace 업로드 완료 | HuggingFace upload complete")
print("https://huggingface.co/cravveo/cravveo-llama-lora")


In [ ]:
# 셀 8: GGUF 변환 (q4_K_M)
# Cell 8: GGUF conversion (q4_K_M)
model.save_pretrained_gguf(
    "cravveo-v6-gguf",
    tokenizer,
    quantization_method="q4_k_m"
)

from google.colab import files
import os

folder = "cravveo-v6-gguf_gguf" if os.path.isdir("cravveo-v6-gguf_gguf") else "cravveo-v6-gguf"
gguf_file = [f for f in os.listdir(folder) if f.endswith(".gguf")][0]
size_gb = os.path.getsize(f"{folder}/{gguf_file}") / 1e9

print(f"파일명: {gguf_file}")
print(f"파일 크기: {size_gb:.2f} GB")

files.download(f"{folder}/{gguf_file}")
print("다운로드 시작 | Download started")
